In [1]:
from google_play_scraper import reviews, Sort
import pandas as pd
import time
from datetime import datetime

print("✅ Libraries loaded!")

✅ Libraries loaded!


In [2]:
BANKS = {
    'Commercial Bank of Ethiopia': 'com.combanketh.mobilebanking',
    'Bank of Abyssinia': 'com.boa.boaMobileBanking',
    'Dashen Bank': 'com.dashen.dashensmart'
}

print("Banks to scrape:")
for bank, app_id in BANKS.items():
    print(f"  {bank}: {app_id}")

Banks to scrape:
  Commercial Bank of Ethiopia: com.combanketh.mobilebanking
  Bank of Abyssinia: com.boa.boaMobileBanking
  Dashen Bank: com.dashen.dashensmart


In [4]:
def scrape_reviews(app_id, bank_name, count=600):
    """
    Scrape reviews from Google Play Store.
    Target 600 per bank to ensure 400+ after cleaning.
    """
    print(f"\nScraping {bank_name}...")
    
    all_reviews = []
    
    try:
        result, _ = reviews(
            app_id,
            lang='en',
            country='et',
            sort=Sort.NEWEST,
            count=count,
            filter_score_with=None
        )
        
        for r in result:
            all_reviews.append({
                'review': r.get('content', ''),
                'rating': r.get('score', None),
                'date': r.get('at', None),
                'bank': bank_name,
                'source': 'Google Play'
            })
        
        print(f"  ✅ Scraped {len(all_reviews)} reviews")
        
    except Exception as e:
        print(f"  ❌ Error scraping {bank_name}: {e}")
    
    time.sleep(2)
    
    return all_reviews

In [5]:
all_data = []

for bank_name, app_id in BANKS.items():
    bank_reviews = scrape_reviews(app_id, bank_name, count=600)
    all_data.extend(bank_reviews)
    print(f"  Total so far: {len(all_data)} reviews")

print(f"\n✅ Total reviews scraped: {len(all_data)}")


Scraping Commercial Bank of Ethiopia...
  ✅ Scraped 600 reviews
  Total so far: 600 reviews

Scraping Bank of Abyssinia...
  ✅ Scraped 600 reviews
  Total so far: 1200 reviews

Scraping Dashen Bank...
  ✅ Scraped 0 reviews
  Total so far: 1200 reviews

✅ Total reviews scraped: 1200


In [6]:
df = pd.DataFrame(all_data)

print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nReviews per bank:")
print(df['bank'].value_counts())
print("\nSample:")
df.head(3)

Shape: (1200, 5)

Columns: ['review', 'rating', 'date', 'bank', 'source']

Reviews per bank:
bank
Commercial Bank of Ethiopia    600
Bank of Abyssinia              600
Name: count, dtype: int64

Sample:


,review,rating,date,bank,source
0,🤙🏼🤙🏼,5,2026-05-16 05:50:50,Commercial Bank of Ethiopia,Google Play
1,worst,1,2026-05-16 02:15:55,Commercial Bank of Ethiopia,Google Play
2,this app very full,5,2026-05-15 23:17:00,Commercial Bank of Ethiopia,Google Play


In [8]:
# Dashen Bank has a different app ID
dashen_ids = [
    'com.dashen.dashensmart',
    'com.dashenbank.mobilebanking',
    'com.dashen.mobile',
    'et.dashenbank.mobilebanking',
    'com.dashenbank',
]

print("Trying different Dashen app IDs...")
for app_id in dashen_ids:
    try:
        result, _ = reviews(
            app_id,
            lang='en',
            country='et',
            sort=Sort.NEWEST,
            count=10
        )
        print(f"✅ FOUND! App ID: {app_id} — {len(result)} reviews")
        break
    except Exception as e:
        print(f"❌ {app_id}: {e}")

Trying different Dashen app IDs...
✅ FOUND! App ID: com.dashen.dashensmart — 0 reviews


In [9]:
# Try scraping Dashen with different settings
from google_play_scraper import reviews, Sort, app

# First check if the app exists
try:
    app_info = app('com.dashen.dashensmart', lang='en', country='et')
    print(f"App found: {app_info['title']}")
    print(f"Total ratings: {app_info.get('ratings', 'N/A')}")
    print(f"Score: {app_info.get('score', 'N/A')}")
except Exception as e:
    print(f"Error: {e}")

# Try with country='us' instead
print("\nTrying country=us...")
try:
    result, _ = reviews(
        'com.dashen.dashensmart',
        lang='en',
        country='us',
        sort=Sort.NEWEST,
        count=100
    )
    print(f"Reviews found: {len(result)}")
except Exception as e:
    print(f"Error: {e}")

# Try with no country filter
print("\nTrying no country filter (country='')...")
try:
    result2, _ = reviews(
        'com.dashen.dashensmart',
        lang='am',
        country='et',
        sort=Sort.NEWEST,
        count=100
    )
    print(f"Amharic reviews found: {len(result2)}")
    if result2:
        print("Sample:", result2[0].get('content', ''))
except Exception as e:
    print(f"Error: {e}")

Error: App not found(404).

Trying country=us...
Reviews found: 0

Trying no country filter (country='')...
Amharic reviews found: 0


In [10]:


print("Trying Awash Bank as Dashen replacement...")

awash_ids = [
    'com.awashbank.mobilebanking',
    'com.awash.mobilebanking', 
    'et.awashbank',
    'com.awashbank',
    'com.awash.banking'
]

for app_id in awash_ids:
    try:
        result, _ = reviews(
            app_id,
            lang='en',
            country='et',
            sort=Sort.NEWEST,
            count=10
        )
        print(f"✅ FOUND! {app_id}: {len(result)} reviews")
        break
    except Exception as e:
        print(f"❌ {app_id}: {e}")

Trying Awash Bank as Dashen replacement...
✅ FOUND! com.awashbank.mobilebanking: 0 reviews


In [11]:
from google_play_scraper import search

# Search for Ethiopian banking apps
print("Searching for Ethiopian banking apps...")
results = search(
    'Ethiopian bank mobile banking',
    lang='en',
    country='et',
    n_hits=20
)

for r in results:
    print(f"App: {r['title']}")
    print(f"  ID: {r['appId']}")
    print(f"  Score: {r.get('score', 'N/A')}")
    print(f"  Ratings: {r.get('ratings', 'N/A')}")
    print()

Searching for Ethiopian banking apps...
App: Commercial Bank of Ethiopia
  ID: com.combanketh.mobilebanking
  Score: 4.286101
  Ratings: N/A

App: EthioDirect
  ID: com.combanketh.ethiodirect
  Score: 4.088398
  Ratings: N/A

App: CBE Connect
  ID: com.eaglelionsystems.connectethiopia
  Score: 4.66
  Ratings: N/A

App: CBEBirr Plus
  ID: prod.cbe.birr
  Score: 4.3931623
  Ratings: N/A

App: CBE Soft Token
  ID: com.cbe.softtoken
  Score: 4.495327
  Ratings: N/A

App: telebirr
  ID: cn.tydic.ethiopay
  Score: 4.4271965
  Ratings: N/A

App: AwashBIRR Pro
  ID: com.sc.awashpay
  Score: 4.3373303
  Ratings: N/A

App: Abyssinia Remit
  ID: com.bankofabyssinia.abyssiniaremit
  Score: 4.28
  Ratings: N/A

App: CBEBirr Agent App
  ID: com.agentapp.agent_app
  Score: 4.35
  Ratings: N/A

App: Fayda ID | የፋይዳ መታወቂያ
  ID: et.fayda.nidfaydaApp
  Score: 4.294953
  Ratings: N/A

App: BoA Mobile
  ID: com.boa.boaMobileBanking
  Score: 4.3919916
  Ratings: N/A

App: Abay Bank Mobile Banking
  ID: com.

In [12]:
# Correct Dashen Bank app ID found!
print("Scraping Dashen Bank with correct app ID...")

dashen_reviews = []
try:
    result, _ = reviews(
        'com.dashen.dashensuperapp',
        lang='en',
        country='et',
        sort=Sort.NEWEST,
        count=600
    )
    
    for r in result:
        dashen_reviews.append({
            'review': r.get('content', ''),
            'rating': r.get('score', None),
            'date': r.get('at', None),
            'bank': 'Dashen Bank',
            'source': 'Google Play'
        })
    
    print(f"✅ Dashen reviews scraped: {len(dashen_reviews)}")
    
except Exception as e:
    print(f"❌ Error: {e}")

Scraping Dashen Bank with correct app ID...
✅ Dashen reviews scraped: 600


In [13]:
# Add Dashen to main dataframe
dashen_df = pd.DataFrame(dashen_reviews)
df = pd.concat([df, dashen_df], ignore_index=True)

print(f"✅ Total reviews: {len(df)}")
print(f"\nReviews per bank:")
print(df['bank'].value_counts())

✅ Total reviews: 1800

Reviews per bank:
bank
Commercial Bank of Ethiopia    600
Bank of Abyssinia              600
Dashen Bank                    600
Name: count, dtype: int64


In [14]:
# ============================================
# PREPROCESSING
# ============================================

print("Starting preprocessing...")
print(f"Shape before cleaning: {df.shape}")

# 1. Remove duplicates
df = df.drop_duplicates(subset=['review', 'bank'])
print(f"After removing duplicates: {df.shape}")

# 2. Drop missing review text or rating
missing_before = df.isnull().sum()
df = df.dropna(subset=['review', 'rating'])
print(f"After dropping missing: {df.shape}")

# 3. Remove empty reviews
df = df[df['review'].str.strip() != '']
print(f"After removing empty reviews: {df.shape}")

# 4. Normalize dates to YYYY-MM-DD
df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')

# 5. Clean rating to integer
df['rating'] = df['rating'].astype(int)

# 6. Reset index
df = df.reset_index(drop=True)

print(f"\n✅ Preprocessing complete!")
print(f"Final shape: {df.shape}")
print(f"\nReviews per bank:")
print(df['bank'].value_counts())
print(f"\nMissing values:")
print(df.isnull().sum())
print(f"\nSample cleaned data:")
df.head(3)

Starting preprocessing...
Shape before cleaning: (1800, 5)
After removing duplicates: (1449, 5)
After dropping missing: (1449, 5)
After removing empty reviews: (1449, 5)

✅ Preprocessing complete!
Final shape: (1449, 5)

Reviews per bank:
bank
Bank of Abyssinia              498
Dashen Bank                    495
Commercial Bank of Ethiopia    456
Name: count, dtype: int64

Missing values:
review    0
rating    0
date      0
bank      0
source    0
dtype: int64

Sample cleaned data:


,review,rating,date,bank,source
0,🤙🏼🤙🏼,5,2026-05-16,Commercial Bank of Ethiopia,Google Play
1,worst,1,2026-05-16,Commercial Bank of Ethiopia,Google Play
2,this app very full,5,2026-05-15,Commercial Bank of Ethiopia,Google Play


In [15]:
# Save cleaned dataset
df.to_csv('../data/raw/all_reviews.csv', index=False)
print(f"✅ Saved to data/raw/all_reviews.csv")
print(f"Total reviews saved: {len(df)}")
print(f"\nColumns: {list(df.columns)}")

✅ Saved to data/raw/all_reviews.csv
Total reviews saved: 1449

Columns: ['review', 'rating', 'date', 'bank', 'source']
